In [2]:
from pathlib import Path

main_directory = Path("../data/crops_types_yearly_capitanata_03035")

tifs_3035 = {}

for d in main_directory.iterdir():
    if d.is_dir():
        year = d.name
        
        lista_tifs = list(d.rglob("*.tif"))

        tifs_3035[year] = [str(tif) for tif in lista_tifs]

In [3]:
from pathlib import Path
import rioxarray

tifs_4326 = {}

for key, value in tifs_3035.items():
    year_file_list = []
    for v in value:
        file_name = v.replace("03035", "4326")
        file_path = Path(file_name)
        
        # Se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # Crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # File GeoTIFF originale (EPSG:3035)
        raster = rioxarray.open_rasterio(v)
        # Riproiezione dell'intero raster in EPSG:4326
        raster_4326 = raster.rio.reproject("EPSG:4326")
        
        raster_4326.rio.to_raster(file_name)
        
        year_file_list.append(file_name)
        
    tifs_4326[key] = year_file_list

In [ ]:
from pathlib import Path
import rasterio
import numpy as np

MAX_SAMPLE_NUMBER = 100

points = []

# Prende la lista di file dell'ultimo gruppo
last_file_list = list(tifs_4326.values())[-1]

for tif in last_file_list:
    with rasterio.open(tif) as dataset:
        full_map = dataset.read(1)

        valid_mask = (full_map > 0) & (full_map < 65534)
        rows, cols = np.where(valid_mask)

        print(full_map.shape)

        indices = np.random.choice(len(rows), size=min(MAX_SAMPLE_NUMBER, len(rows)), replace=False)

        for index in indices:
            lat, lon = dataset.xy(cols[index], rows[index])
            points.append({
                "lat": lat.item(),
                "lon": lon.item(),
                "code": full_map[rows[index], cols[index]].item()
            })

/tmp/ipykernel_15787/3210782724.py:16: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  full_map = dataset.read(1)


(9176, 12240)
(9344, 12442)
(9407, 12327)
(9243, 12135)


In [62]:
import pandas as pd
df = pd.read_json('../data/points.json')
conteggio = df['code'].value_counts()
print(conteggio)

code
2200    166
1110    107
1120     36
1210     20
2310     17
2100     14
3100     12
1220     10
1410      8
1150      5
1130      2
1310      2
1430      1
Name: count, dtype: int64


In [61]:
import json
from pathlib import Path

# Percorso del file JSON nella cartella di lavoro
json_path = Path("../data/points.json")

# Salva i punti (formato: [["nome_file", lon, lat], ...])
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(points, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(points)} punti in {json_path.resolve()}")

✅ Salvati 400 punti in /home/gmatteo/Developer/unibs/mldm/crop-spatial-classification/data/points.json
